In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import json
import os
from datetime import datetime

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


In [2]:


DATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data"
MODELS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\models"
METRICS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\metrics"
REPORTS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\reports"
DOCS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs"
RANDOM_SEED = 42

for path in [MODELS_PATH, METRICS_PATH, REPORTS_PATH, DOCS_PATH]:
    os.makedirs(path, exist_ok=True)


In [5]:

preprocessor = joblib.load("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\models\\processingpreprocessing_pipeline.pkl")

X_train_raw = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_train.csv")
X_val_raw   = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_val.csv")
X_test_raw  = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_test.csv")

y_co2_train = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_co2_train.csv").values.ravel()
y_co2_val   = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_co2_val.csv").values.ravel()
y_co2_test  = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_co2_test.csv").values.ravel()

print("✓ Data loaded successfully")


✓ Data loaded successfully


In [25]:

X_train = preprocessor.transform(X_train_raw)
X_val   = preprocessor.transform(X_val_raw)
X_test  = preprocessor.transform(X_test_raw)

print("Transformed shapes:", X_train.shape, X_val.shape, X_test.shape)


Transformed shapes: (424200, 23) (60600, 23) (121200, 23)


In [26]:
param_grid = {
    'n_estimators': [100],
    'max_depth': [4, 6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_co2_train)

best_params = grid_search.best_params_
best_params.update({
    'objective': 'reg:squarederror',
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
    'verbosity': 0
})

print("Best params:", best_params)


Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}


In [28]:
xgb_final = xgb.XGBRegressor(**best_params)


xgb_final.fit(
    X_train,
    y_co2_train
)

print("Model trained successfully")



Model trained successfully


In [29]:
y_train_pred = xgb_final.predict(X_train)
y_val_pred   = xgb_final.predict(X_val)
y_test_pred  = xgb_final.predict(X_test)


In [30]:
def calculate_metrics(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred)
    }


In [31]:
train_metrics = calculate_metrics(y_co2_train, y_train_pred)
val_metrics   = calculate_metrics(y_co2_val, y_val_pred)
test_metrics  = calculate_metrics(y_co2_test, y_test_pred)

metrics_df = pd.DataFrame([
    {'set': 'train', **train_metrics},
    {'set': 'val', **val_metrics},
    {'set': 'test', **test_metrics}
])

metrics_df


,set,mae,rmse,r2
0,train,0.000255,0.000338,1.0
1,val,0.000255,0.000340,1.0
2,test,0.000255,0.000338,1.0


In [35]:
joblib.dump(
    xgb_final,
    "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\reports\\xgb_co2_optimized.joblib"
)

metrics_df.to_csv(
    "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\reports\\co2_metrics.csv",
    index=False
)

print("Model and metrics saved")


Model and metrics saved


In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_train_raw.columns,
    'importance': xgb_final.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance.head(10)

feature_importance.to_csv(
    "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\reports\\xgb_feature_importance.csv",
    index=False
)


In [36]:
summary = f"""
EcoPackAI - XGBoost CO2 Prediction Summary

Train R2: {train_metrics['r2']:.4f}
Val R2:   {val_metrics['r2']:.4f}
Test R2:  {test_metrics['r2']:.4f}

Training Date: {datetime.now()}
"""

with open("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs\\xgb_co2_training_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)

print("Summary saved")


Summary saved


In [39]:
import json
from datetime import datetime
import os

# ============================================
# SAVE XGBOOST CO₂ MODEL CONFIGURATION
# ============================================

CONFIG_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\models"
os.makedirs(CONFIG_PATH, exist_ok=True)

xgb_co2_config = {
    "model_name": "XGBRegressor",
    "target": "co2_emission_per_kg",
    "task": "regression",
    "optimized": True,
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "hyperparameters": best_params,

    "training_info": {
        "n_estimators": best_params.get("n_estimators"),
        "max_depth": best_params.get("max_depth"),
        "learning_rate": best_params.get("learning_rate"),
        "subsample": best_params.get("subsample"),
        "colsample_bytree": best_params.get("colsample_bytree"),
        "objective": "reg:squarederror",
        "eval_metric": "rmse"
    },

    "notes": [
        "Model trained on integrated product-material dataset",
        "Hyperparameters tuned using validation RMSE",
        "Used for production CO₂ emission prediction"
    ]
}

# Save config file
config_file = os.path.join("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\reports\\xgb_co2_config.json")
with open(config_file, "w", encoding="utf-8") as f:
    json.dump(xgb_co2_config, f, indent=2)

print(f"✓ XGBoost CO₂ config saved: {config_file}")


✓ XGBoost CO₂ config saved: C:\Users\sneha\Desktop\ecopackai\Data\ml\reports\xgb_co2_config.json
